In [1]:
import sys
print(sys.executable)

/Users/liudvikas/Programming/poseResearch/pose_estimation/.venv/bin/python


In [2]:
from mmpose.apis import MMPoseInferencer

img_path = 'image.png'   # replace this with your own image path

# instantiate the inferencer using the model alias
inferencer = MMPoseInferencer(pose3d="configs/body_3d_keypoint/motionbert/h36m/motionbert_dstformer-ft-243frm_8xb32-120e_h36m.py", pose3d_weights="models/motionbert_ft_h36m-d80af323_20230531.pth", device='cpu')

# The MMPoseInferencer API employs a lazy inference approach,
# creating a prediction generator when given input
result_generator = inferencer(img_path, vis_out_dir='./vis_out', pred_out_dir='./pred_out', radius=4, thickness=2, num_instances=1)
result = next(result_generator)

AttributeError: module 'mmcv' has no attribute '__version__'

In [2]:
from mmcv.image import imread
import cv2
import logging
import mimetypes
import os
import time
from argparse import ArgumentParser
from functools import partial

import cv2
import json_tricks as json
import mmcv
import mmengine
import numpy as np
from mmengine.logging import print_log

from mmpose.apis import inference_topdown, init_model, inference_bottomup, inference_pose_lifter_model, extract_pose_sequence, _track_by_iou, _track_by_oks, convert_keypoint_definition
from mmpose.registry import VISUALIZERS
from mmpose.structures import PoseDataSample, merge_data_samples, split_instances
from mmdet.apis import inference_detector, init_detector
from mmpose.evaluation.functional import nms
from mmpose.utils import adapt_mmdet_pipeline
from mmpose.models.pose_estimators.topdown import TopdownPoseEstimator
from mmpose.models.pose_estimators import PoseLifter

has_mmdet = True

# Default arguments
class Args:
    def __init__(self):
        self.show = False
        self.output_root = 'res'
        self.save_predictions = True
        self.device = 'cpu'
        self.det_cat_id = 0
        self.bbox_thr = 0.3
        self.nms_thr = 0.3
        self.kpt_thr = 0.3
        self.draw_heatmap = False
        self.show_kpt_idx = False
        self.skeleton_style = 'mmpose'
        self.radius = 3
        self.thickness = 1
        self.show_interval = 0
        self.alpha = 0.8
        self.draw_bbox = False
        self.input = 'test-video-small.mp4'
        self.disable_rebase_keypoint = False
        self.use_oks_tracking = False
        self.tracking_thr = 0.3
        self.disable_norm_pose_2d = False
        self.num_instances = 3
        self.pred_save_path = "predictions"

args = Args()

AttributeError: module 'mmcv' has no attribute '__version__'

In [24]:
def process_one_image(detector, frame, frame_idx, pose_estimator,
                      pose_est_results_last, pose_est_results_list, next_id,
                      pose_lifter, visualize_frame, visualizer):
    """Visualize detected and predicted keypoints of one image.

    Pipeline of this function:

                              frame
                                |
                                V
                        +-----------------+
                        |     detector    |
                        +-----------------+
                                |  det_result
                                V
                        +-----------------+
                        |  pose_estimator |
                        +-----------------+
                                |  pose_est_results
                                V
            +--------------------------------------------+
            |  convert 2d kpts into pose-lifting format  |
            +--------------------------------------------+
                                |  pose_est_results_list
                                V
                    +-----------------------+
                    | extract_pose_sequence |
                    +-----------------------+
                                |  pose_seq_2d
                                V
                         +-------------+
                         | pose_lifter |
                         +-------------+
                                |  pose_lift_results
                                V
                       +-----------------+
                       | post-processing |
                       +-----------------+
                                |  pred_3d_data_samples
                                V
                         +------------+
                         | visualizer |
                         +------------+

    Args:
        args (Argument): Custom command-line arguments.
        detector (mmdet.BaseDetector): The mmdet detector.
        frame (np.ndarray): The image frame read from input image or video.
        frame_idx (int): The index of current frame.
        pose_estimator (TopdownPoseEstimator): The pose estimator for 2d pose.
        pose_est_results_last (list(PoseDataSample)): The results of pose
            estimation from the last frame for tracking instances.
        pose_est_results_list (list(list(PoseDataSample))): The list of all
            pose estimation results converted by
            ``convert_keypoint_definition`` from previous frames. In
            pose-lifting stage it is used to obtain the 2d estimation sequence.
        next_id (int): The next track id to be used.
        pose_lifter (PoseLifter): The pose-lifter for estimating 3d pose.
        visualize_frame (np.ndarray): The image for drawing the results on.
        visualizer (Visualizer): The visualizer for visualizing the 2d and 3d
            pose estimation results.

    Returns:
        pose_est_results (list(PoseDataSample)): The pose estimation result of
            the current frame.
        pose_est_results_list (list(list(PoseDataSample))): The list of all
            converted pose estimation results until the current frame.
        pred_3d_instances (InstanceData): The result of pose-lifting.
            Specifically, the predicted keypoints and scores are saved at
            ``pred_3d_instances.keypoints`` and
            ``pred_3d_instances.keypoint_scores``.
        next_id (int): The next track id to be used.
    """
    pose_lift_dataset = pose_lifter.cfg.test_dataloader.dataset
    pose_lift_dataset_name = pose_lifter.dataset_meta['dataset_name']

    # First stage: conduct 2D pose detection in a Topdown manner
    # use detector to obtain person bounding boxes
    det_result = inference_detector(detector, frame)
    pred_instance = det_result.pred_instances.cpu().numpy()

    # filter out the person instances with category and bbox threshold
    # e.g. 0 for person in COCO
    bboxes = pred_instance.bboxes
    bboxes = bboxes[np.logical_and(pred_instance.labels == args.det_cat_id,
                                   pred_instance.scores > args.bbox_thr)]

    # estimate pose results for current image
    pose_est_results = inference_topdown(pose_estimator, frame, bboxes)

    if args.use_oks_tracking:
        _track = partial(_track_by_oks)
    else:
        _track = _track_by_iou

    pose_det_dataset_name = pose_estimator.dataset_meta['dataset_name']
    pose_est_results_converted = []

    # convert 2d pose estimation results into the format for pose-lifting
    # such as changing the keypoint order, flipping the keypoint, etc.
    for i, data_sample in enumerate(pose_est_results):
        pred_instances = data_sample.pred_instances.cpu().numpy()
        keypoints = pred_instances.keypoints
        # calculate area and bbox
        if 'bboxes' in pred_instances:
            areas = np.array([(bbox[2] - bbox[0]) * (bbox[3] - bbox[1])
                              for bbox in pred_instances.bboxes])
            pose_est_results[i].pred_instances.set_field(areas, 'areas')
        else:
            areas, bboxes = [], []
            for keypoint in keypoints:
                xmin = np.min(keypoint[:, 0][keypoint[:, 0] > 0], initial=1e10)
                xmax = np.max(keypoint[:, 0])
                ymin = np.min(keypoint[:, 1][keypoint[:, 1] > 0], initial=1e10)
                ymax = np.max(keypoint[:, 1])
                areas.append((xmax - xmin) * (ymax - ymin))
                bboxes.append([xmin, ymin, xmax, ymax])
            pose_est_results[i].pred_instances.areas = np.array(areas)
            pose_est_results[i].pred_instances.bboxes = np.array(bboxes)

        # track id
        track_id, pose_est_results_last, _ = _track(data_sample,
                                                    pose_est_results_last,
                                                    args.tracking_thr)
        if track_id == -1:
            if np.count_nonzero(keypoints[:, :, 1]) >= 3:
                track_id = next_id
                next_id += 1
            else:
                # If the number of keypoints detected is small,
                # delete that person instance.
                keypoints[:, :, 1] = -10
                pose_est_results[i].pred_instances.set_field(
                    keypoints, 'keypoints')
                pose_est_results[i].pred_instances.set_field(
                    pred_instances.bboxes * 0, 'bboxes')
                pose_est_results[i].set_field(pred_instances, 'pred_instances')
                track_id = -1
        pose_est_results[i].set_field(track_id, 'track_id')

        # convert keypoints for pose-lifting
        pose_est_result_converted = PoseDataSample()
        pose_est_result_converted.set_field(
            pose_est_results[i].pred_instances.clone(), 'pred_instances')
        pose_est_result_converted.set_field(
            pose_est_results[i].gt_instances.clone(), 'gt_instances')
        keypoints = convert_keypoint_definition(keypoints,
                                                pose_det_dataset_name,
                                                pose_lift_dataset_name)
        pose_est_result_converted.pred_instances.set_field(
            keypoints, 'keypoints')
        pose_est_result_converted.set_field(pose_est_results[i].track_id,
                                            'track_id')
        pose_est_results_converted.append(pose_est_result_converted)

    pose_est_results_list.append(pose_est_results_converted.copy())

    # Second stage: Pose lifting
    # extract and pad input pose2d sequence
    pose_seq_2d = extract_pose_sequence(
        pose_est_results_list,
        frame_idx=frame_idx,
        causal=pose_lift_dataset.get('causal', False),
        seq_len=pose_lift_dataset.get('seq_len', 1),
        step=pose_lift_dataset.get('seq_step', 1))

    # conduct 2D-to-3D pose lifting
    norm_pose_2d = not args.disable_norm_pose_2d
    pose_lift_results = inference_pose_lifter_model(
        pose_lifter,
        pose_seq_2d,
        image_size=visualize_frame.shape[:2],
        norm_pose_2d=norm_pose_2d)

    # post-processing
    for idx, pose_lift_result in enumerate(pose_lift_results):
        pose_lift_result.track_id = pose_est_results[idx].get('track_id', 1e4)

        pred_instances = pose_lift_result.pred_instances
        keypoints = pred_instances.keypoints
        keypoint_scores = pred_instances.keypoint_scores
        if keypoint_scores.ndim == 3:
            keypoint_scores = np.squeeze(keypoint_scores, axis=1)
            pose_lift_results[
                idx].pred_instances.keypoint_scores = keypoint_scores
        if keypoints.ndim == 4:
            keypoints = np.squeeze(keypoints, axis=1)

        keypoints = keypoints[..., [0, 2, 1]]
        keypoints[..., 0] = -keypoints[..., 0]
        keypoints[..., 2] = -keypoints[..., 2]

        # rebase height (z-axis)
        if not args.disable_rebase_keypoint:
            keypoints[..., 2] -= np.min(
                keypoints[..., 2], axis=-1, keepdims=True)

        pose_lift_results[idx].pred_instances.keypoints = keypoints

    pose_lift_results = sorted(
        pose_lift_results, key=lambda x: x.get('track_id', 1e4))

    pred_3d_data_samples = merge_data_samples(pose_lift_results)
    det_data_sample = merge_data_samples(pose_est_results)
    pred_3d_instances = pred_3d_data_samples.get('pred_instances', None)

    if args.num_instances < 0:
        args.num_instances = len(pose_lift_results)
    print(pose_lift_results)
    print(len(pose_lift_results))

    # Visualization
    if visualizer is not None:
        visualizer.add_datasample(
            'result',
            visualize_frame,
            data_sample=pred_3d_data_samples,
            det_data_sample=det_data_sample,
            draw_gt=False,
            dataset_2d=pose_det_dataset_name,
            dataset_3d=pose_lift_dataset_name,
            show=args.show,
            draw_bbox=True,
            kpt_thr=args.kpt_thr,
            num_instances=args.num_instances,
            wait_time=args.show_interval)

    return pose_est_results, pose_est_results_list, pred_3d_instances, next_id

In [13]:
det_config = 'configs/rtmdet_m_640-8xb32_coco-person.py'
det_checkpoint = 'models/rtmdet_m_8xb32-100e_coco-obj365-person-235e8209.pth'

pose2d_config = 'configs/body_2d_keypoint/rtmpose/body8/rtmpose-m_8xb256-420e_body8-256x192.py'
pose2d_weights = 'models/rtmpose-m_simcc-body7_pt-body7_420e-256x192-e48f03d0_20230504.pth'

pose3d_config = 'configs/body_3d_keypoint/video_pose_lift/h36m/video-pose-lift_tcn-243frm-supv-cpn-ft_8xb128-200e_h36m.py'
pose3d_weights = 'models/videopose_h36m_243frames_fullconv_supervised_cpn_ft-88f5abbb_20210527.pth'


In [17]:
# build detector
detector = init_detector(
    det_config, det_checkpoint, device=args.device)
detector.cfg = adapt_mmdet_pipeline(detector.cfg)
pose_estimator = init_model(
        pose2d_config,
        pose2d_weights,
        device=args.device.lower())

assert isinstance(pose_estimator, TopdownPoseEstimator), 'Only "TopDown"' \
        'model is supported for the 1st stage (2D pose detection)'

det_kpt_color = pose_estimator.dataset_meta.get('keypoint_colors', None)
det_dataset_skeleton = pose_estimator.dataset_meta.get(
    'skeleton_links', None)
det_dataset_link_color = pose_estimator.dataset_meta.get(
    'skeleton_link_colors', None)

pose_lifter = init_model(
        pose3d_config,
        pose3d_weights,
        device=args.device.lower())

assert isinstance(pose_lifter, PoseLifter), \
        'Only "PoseLifter" model is supported for the 2nd stage ' \
        '(2D-to-3D lifting)'

pose_lifter.cfg.visualizer.radius = args.radius
pose_lifter.cfg.visualizer.line_width = args.thickness
pose_lifter.cfg.visualizer.det_kpt_color = det_kpt_color
pose_lifter.cfg.visualizer.det_dataset_skeleton = det_dataset_skeleton
pose_lifter.cfg.visualizer.det_dataset_link_color = det_dataset_link_color
visualizer = VISUALIZERS.build(pose_lifter.cfg.visualizer)

# the dataset_meta is loaded from the checkpoint
visualizer.set_dataset_meta(pose_lifter.dataset_meta)

input_type = mimetypes.guess_type(args.input)[0].split('/')[0]

mmengine.mkdir_or_exist(args.output_root)
output_file = os.path.join(args.output_root,
                        os.path.basename(args.input))
save_output = True

if args.save_predictions:
        assert args.output_root != ''
        args.pred_save_path = f'{args.output_root}/results_' \
                f'{os.path.splitext(os.path.basename(args.input))[0]}.json'
fourcc = cv2.VideoWriter_fourcc(*'mp4v')

# Pose estimation results
pose_est_results_list = []
pred_instances_list = []




Loads checkpoint by local backend from path: models/rtmdet_m_8xb32-100e_coco-obj365-person-235e8209.pth


/Users/liudvikas/miniconda3/envs/openmmlab/lib/python3.8/site-packages/mmengine/runner/checkpoint.py:347: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.lo

Loads checkpoint by local backend from path: models/rtmpose-m_simcc-body7_pt-body7_420e-256x192-e48f03d0_20230504.pth
Loads checkpoint by local backend from path: models/videopose_h36m_243frames_fullconv_supervised_cpn_ft-88f5abbb_20210527.pth


/Users/liudvikas/miniconda3/envs/openmmlab/lib/python3.8/site-packages/mmengine/utils/manager.py:113: UserWarning: <class 'mmpose.visualization.local_visualizer_3d.Pose3dLocalVisualizer'> instance named of visualizer has been created, the method `get_instance` should not accept any other arguments
  warnings.warn(


In [25]:
next_id = 0
pose_est_results = []

if args.input == 'webcam':
    video = cv2.VideoCapture(0)
else:
    video = cv2.VideoCapture(args.input)

(major_ver, minor_ver, subminor_ver) = (cv2.__version__).split('.')
if int(major_ver) < 3:
    fps = video.get(cv2.cv.CV_CAP_PROP_FPS)
else:
    fps = video.get(cv2.CAP_PROP_FPS)

video_writer = None
frame_idx = 0

while video.isOpened():
    success, frame = video.read()
    frame_idx += 1

    if not success:
        break

    pose_est_results_last = pose_est_results

    # First stage: 2D pose detection
    # make person results for current image
    (pose_est_results, pose_est_results_list, pred_3d_instances,
        next_id) = process_one_image(
            detector=detector,
            frame=frame,
            frame_idx=frame_idx,
            pose_estimator=pose_estimator,
            pose_est_results_last=pose_est_results_last,
            pose_est_results_list=pose_est_results_list,
            next_id=next_id,
            pose_lifter=pose_lifter,
            visualize_frame=mmcv.bgr2rgb(frame),
            visualizer=visualizer)

    if args.save_predictions:
        # save prediction results
        pred_instances_list.append(
            dict(
                frame_id=frame_idx,
                instances=split_instances(pred_3d_instances)))

    if save_output:
        frame_vis = visualizer.get_image()
        if video_writer is None:
            # the size of the image with visualization may vary
            # depending on the presence of heatmaps
            video_writer = cv2.VideoWriter(output_file, fourcc, fps,
                                            (frame_vis.shape[1],
                                            frame_vis.shape[0]))

        video_writer.write(mmcv.rgb2bgr(frame_vis))

    if args.show:
        # press ESC to exit
        if cv2.waitKey(5) & 0xFF == 27:
            break
        time.sleep(args.show_interval)

video.release()

if video_writer:
    video_writer.release()

[<PoseDataSample(

    META INFORMATION
    target_root: array([[0., 0., 0.]], dtype=float32)
    flip_indices: [0, 4, 5, 6, 1, 2, 3, 7, 8, 9, 10, 14, 15, 16, 11, 12, 13]

    DATA FIELDS
    gt_instance_labels: <InstanceData(
        
            META INFORMATION
        
            DATA FIELDS
            lifting_target_label: tensor([[[0., 0., 0.],
                         [0., 0., 0.],
                         [0., 0., 0.],
                         [0., 0., 0.],
                         [0., 0., 0.],
                         [0., 0., 0.],
                         [0., 0., 0.],
                         [0., 0., 0.],
                         [0., 0., 0.],
                         [0., 0., 0.],
                         [0., 0., 0.],
                         [0., 0., 0.],
                         [0., 0., 0.],
                         [0., 0., 0.],
                         [0., 0., 0.],
                         [0., 0., 0.],
                         [0., 0., 0.]]])
            lifting

ValueError: cannot reshape array of size 18662400 into shape (1440,12960,3)

In [10]:
if args.save_predictions:
    with open(args.pred_save_path, 'w') as f:
        json.dump(
            dict(
                meta_info=pose_lifter.dataset_meta,
                instance_info=pred_instances_list),
            f,
            indent='\t')
    print(f'predictions have been saved at {args.pred_save_path}')

input_type = input_type.replace('webcam', 'video')
print_log(
    f'the output {input_type} has been saved at {output_file}',
    logger='current',
    level=logging.INFO)

predictions have been saved at res/results_test-video-small.json
05/21 01:35:29 - mmengine - INFO - the output video has been saved at res/test-video-small.mp4
